# Backtest Statistical Analysis

Step-by-step analysis of backtest round-trip trade results.

**What we'll cover:**
1. Load & inspect trade data
2. Core metrics (win rate, EV, reward-to-risk)
3. Stress tests (monthly, quarterly, market regime breakdowns)
4. Risk metrics (max consecutive losses, max drawdown, std dev)
5. Visualizations (equity curve, P&L distribution, monthly performance)

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pretty display settings
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 120
plt.style.use("seaborn-v0_8-whitegrid")

# ── CONFIG: Change these to match your backtest ──
TRADES_CSV = "../../backtest/report/csv/backtest_trades_BTCUSDT_15m.csv"   # path to round-trip CSV
INITIAL_BALANCE = 100_000  # starting balance used in the backtest

## Step 1: Load & Inspect Data

Load the round-trip trades CSV and get a feel for the data.

In [ ]:
df = pd.read_csv(TRADES_CSV)
df["entry_time"] = pd.to_datetime(df["entry_time"])
df["exit_time"] = pd.to_datetime(df["exit_time"])

print(f"Total trades: {len(df)}")
print(f"Date range:   {df['entry_time'].min()} → {df['exit_time'].max()}")
print(f"Columns:      {list(df.columns)}")
df.head()

In [ ]:
# Quick summary of the P&L column
df[["pnl", "pnl_pct", "hold_duration_hours"]].describe()

## Step 2: Core Metrics

The fundamental numbers every trader needs:

| Metric | Formula |
|--------|---------|
| **Win Rate** | wins / total trades |
| **Avg Win** | mean P&L of winning trades |
| **Avg Loss** | mean P&L of losing trades |
| **EV per Trade** | (win_rate × avg_win) + (loss_rate × avg_loss) |
| **Reward-to-Risk** | avg_win / \|avg_loss\| |
| **Profit Factor** | gross_profit / gross_loss |

In [ ]:
wins = df[df["pnl"] > 0]
losses = df[df["pnl"] <= 0]

total_trades = len(df)
win_count = len(wins)
loss_count = len(losses)
win_rate = win_count / total_trades * 100
loss_rate = 100 - win_rate

avg_win = wins["pnl"].mean() if win_count > 0 else 0
avg_loss = losses["pnl"].mean() if loss_count > 0 else 0

# EV = (win_rate × avg_win) + (loss_rate × avg_loss)
ev_per_trade = (win_rate / 100 * avg_win) + (loss_rate / 100 * avg_loss)

# Reward-to-Risk = avg_win / |avg_loss|
reward_to_risk = abs(avg_win / avg_loss) if avg_loss != 0 else float("inf")

# Profit Factor = gross_profit / gross_loss
gross_profit = wins["pnl"].sum() if win_count > 0 else 0
gross_loss = abs(losses["pnl"].sum()) if loss_count > 0 else 0
profit_factor = gross_profit / gross_loss if gross_loss > 0 else float("inf")

total_pnl = df["pnl"].sum()

# Display as a clean table
core_metrics = pd.DataFrame({
    "Metric": [
        "Total Trades", "Winning Trades", "Losing Trades",
        "Win Rate (%)", "Avg Win ($)", "Avg Loss ($)",
        "EV per Trade ($)", "Reward-to-Risk", "Profit Factor",
        "Total P&L ($)",
    ],
    "Value": [
        f"{total_trades}", f"{win_count}", f"{loss_count}",
        f"{win_rate:.2f}%", f"${avg_win:,.2f}", f"${avg_loss:,.2f}",
        f"${ev_per_trade:,.2f}", f"{reward_to_risk:.2f}", f"{profit_factor:.2f}",
        f"${total_pnl:,.2f}",
    ],
})
core_metrics.style.hide(axis="index")

### How to interpret

- **Win rate > 50%** is good, but only matters in combination with reward-to-risk
- **EV per trade > 0** means the strategy is profitable on average
- **Reward-to-Risk > 1** means wins are bigger than losses (you can be profitable even with <50% win rate)
- **Profit Factor > 1.5** is considered a solid edge

**The key relationship:**
- If R:R is low (e.g. 0.5), you need a very high win rate (~67%+) to break even
- If R:R is high (e.g. 2.0), you only need ~33% win rate to break even
- Break-even win rate = 1 / (1 + R:R)

In [ ]:
# Break-even analysis
breakeven_wr = 1 / (1 + reward_to_risk) * 100
print(f"Your R:R = {reward_to_risk:.2f}")
print(f"Break-even win rate needed = {breakeven_wr:.1f}%")
print(f"Your actual win rate       = {win_rate:.1f}%")
print(f"Gap                        = {win_rate - breakeven_wr:+.1f}% {'(EDGE EXISTS)' if win_rate > breakeven_wr else '(NO EDGE - need higher R:R or win rate)'}")

## Step 3: Stress Tests

Break down performance by time period and market condition to see if the edge is consistent or driven by a few lucky months.

### 3a. Monthly Breakdown

In [ ]:
df["_month"] = df["exit_time"].dt.to_period("M")

monthly = df.groupby("_month").agg(
    trades=("pnl", "count"),
    wins=("pnl", lambda x: (x > 0).sum()),
    pnl=("pnl", "sum"),
    avg_pnl=("pnl", "mean"),
).reset_index()

monthly["win_rate"] = (monthly["wins"] / monthly["trades"] * 100).round(1)
monthly["month"] = monthly["_month"].astype(str)
monthly[["month", "trades", "wins", "win_rate", "pnl", "avg_pnl"]]

### 3b. Quarterly Breakdown

In [ ]:
df["_quarter"] = df["exit_time"].dt.to_period("Q")

quarterly = df.groupby("_quarter").agg(
    trades=("pnl", "count"),
    wins=("pnl", lambda x: (x > 0).sum()),
    pnl=("pnl", "sum"),
).reset_index()

quarterly["win_rate"] = (quarterly["wins"] / quarterly["trades"] * 100).round(1)
quarterly["quarter"] = quarterly["_quarter"].astype(str)
quarterly[["quarter", "trades", "wins", "win_rate", "pnl"]]

### 3c. Market Regime Breakdown

Classify each trade into a market regime based on price movement between consecutive trades:
- **TRENDING_UP**: entry price rose >1% vs previous trade's entry
- **TRENDING_DOWN**: entry price fell >1%
- **RANGING**: within +/-1%

This helps answer: *"Does the strategy work better in trends or ranges?"*

In [ ]:
tmp = df.sort_values("entry_time").reset_index(drop=True)
prev_price = tmp["entry_price"].shift(1)
pct_change = (tmp["entry_price"] - prev_price) / prev_price * 100

conditions = [pct_change > 1.0, pct_change < -1.0]
choices = ["TRENDING_UP", "TRENDING_DOWN"]
tmp["regime"] = np.select(conditions, choices, default="RANGING")
tmp.loc[0, "regime"] = "RANGING"  # first trade has no prior

regime = tmp.groupby("regime").agg(
    trades=("pnl", "count"),
    wins=("pnl", lambda x: (x > 0).sum()),
    pnl=("pnl", "sum"),
    avg_pnl=("pnl", "mean"),
).reset_index()

regime["win_rate"] = (regime["wins"] / regime["trades"] * 100).round(1)
regime

### 3d. Exit Reason Breakdown

How are trades exiting? This tells you if your TP/SL levels are well-calibrated.

In [ ]:
exit_stats = df.groupby("exit_reason").agg(
    trades=("pnl", "count"),
    avg_pnl=("pnl", "mean"),
    total_pnl=("pnl", "sum"),
).sort_values("trades", ascending=False)

exit_stats

## Step 4: Risk Metrics

These metrics help you understand the *worst-case* scenarios and overall risk profile.

In [ ]:
# ── Max consecutive wins/losses ──
def max_consecutive(series, value):
    max_count = current = 0
    for v in series:
        if v == value:
            current += 1
            max_count = max(max_count, current)
        else:
            current = 0
    return max_count

is_win = (df["pnl"] > 0).astype(int)
max_consec_wins = max_consecutive(is_win, 1)
max_consec_losses = max_consecutive(is_win, 0)

# ── Equity curve & drawdown ──
equity = [INITIAL_BALANCE]
for pnl in df["pnl"]:
    equity.append(equity[-1] + pnl)

peak = equity[0]
drawdowns = []
max_dd = max_dd_val = 0.0
for val in equity:
    if val > peak:
        peak = val
    dd = (peak - val) / peak if peak > 0 else 0
    drawdowns.append(dd * 100)
    if dd > max_dd:
        max_dd = dd
        max_dd_val = peak - val

# ── Return distribution stats ──
returns_pct = df["pnl_pct"].values
std_dev = np.std(returns_pct, ddof=1) if len(returns_pct) > 1 else 0
mean_ret = np.mean(returns_pct)
sharpe = mean_ret / std_dev if std_dev > 0 else 0

neg_returns = returns_pct[returns_pct < 0]
downside_std = np.std(neg_returns, ddof=1) if len(neg_returns) > 1 else 0
sortino = mean_ret / downside_std if downside_std > 0 else 0

var_95 = np.percentile(returns_pct, 5) if len(returns_pct) >= 5 else min(returns_pct)

# Display
risk_metrics = pd.DataFrame({
    "Metric": [
        "Max Consecutive Wins", "Max Consecutive Losses",
        "Max Drawdown ($)", "Max Drawdown (%)",
        "Std Dev of Returns (%)", "Sharpe Ratio (trade-level)",
        "Sortino Ratio (trade-level)", "Value-at-Risk 95% (%)",
    ],
    "Value": [
        f"{max_consec_wins}", f"{max_consec_losses}",
        f"${max_dd_val:,.2f}", f"{max_dd * 100:.2f}%",
        f"{std_dev:.2f}%", f"{sharpe:.3f}",
        f"{sortino:.3f}", f"{var_95:.2f}%",
    ],
})
risk_metrics.style.hide(axis="index")

### How to interpret risk metrics

- **Max Consecutive Losses**: How many losses in a row you should mentally prepare for. If this is 5+, you need strong discipline.
- **Max Drawdown**: The worst peak-to-trough decline. A 20%+ drawdown is psychologically tough to endure live.
- **Sharpe Ratio**: Return per unit of risk. Above 1.0 is good, above 2.0 is excellent.
- **Sortino Ratio**: Like Sharpe but only penalizes downside volatility. Better metric for asymmetric strategies.
- **VaR 95%**: "In the worst 5% of trades, you'll lose at least X%". This is your tail risk.

## Step 5: Visualizations

### 5a. Equity Curve

The most important chart — shows how your balance evolved over time.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

# Equity curve
ax1.plot(range(len(equity)), equity, linewidth=1.5, color="#2196F3")
ax1.axhline(y=INITIAL_BALANCE, color="gray", linestyle="--", alpha=0.5, label="Initial Balance")
ax1.fill_between(range(len(equity)), equity, INITIAL_BALANCE,
                  where=[e >= INITIAL_BALANCE for e in equity], alpha=0.15, color="green")
ax1.fill_between(range(len(equity)), equity, INITIAL_BALANCE,
                  where=[e < INITIAL_BALANCE for e in equity], alpha=0.15, color="red")
ax1.set_title("Equity Curve", fontsize=14, fontweight="bold")
ax1.set_ylabel("Balance ($)")
ax1.legend()

# Drawdown subplot
ax2.fill_between(range(len(drawdowns)), drawdowns, color="red", alpha=0.3)
ax2.plot(range(len(drawdowns)), drawdowns, color="red", linewidth=0.8)
ax2.set_title("Drawdown (%)", fontsize=11)
ax2.set_xlabel("Trade #")
ax2.set_ylabel("DD %")
ax2.invert_yaxis()  # drawdown goes down

fig.tight_layout()
plt.show()

### 5b. Win/Loss Distribution

Histogram showing the spread of trade outcomes. Ideally you want the green (wins) clustered to the right and losses tightly clustered near zero.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# P&L in dollars
bin_edges = np.linspace(df["pnl"].min(), df["pnl"].max(), 25)
ax1.hist(wins["pnl"], bins=bin_edges, color="#4CAF50", alpha=0.7, label=f"Wins ({len(wins)})")
ax1.hist(losses["pnl"], bins=bin_edges, color="#F44336", alpha=0.7, label=f"Losses ({len(losses)})")
ax1.axvline(x=0, color="black", linewidth=0.8)
ax1.axvline(x=df["pnl"].mean(), color="blue", linestyle="--", label=f"Mean: ${df['pnl'].mean():,.0f}")
ax1.set_title("P&L Distribution ($)", fontsize=13, fontweight="bold")
ax1.set_xlabel("P&L ($)")
ax1.set_ylabel("Frequency")
ax1.legend()

# P&L in percent
bin_edges_pct = np.linspace(df["pnl_pct"].min(), df["pnl_pct"].max(), 25)
ax2.hist(wins["pnl_pct"], bins=bin_edges_pct, color="#4CAF50", alpha=0.7, label="Wins")
ax2.hist(losses["pnl_pct"], bins=bin_edges_pct, color="#F44336", alpha=0.7, label="Losses")
ax2.axvline(x=0, color="black", linewidth=0.8)
ax2.axvline(x=df["pnl_pct"].mean(), color="blue", linestyle="--", label=f"Mean: {df['pnl_pct'].mean():.1f}%")
ax2.set_title("P&L Distribution (%)", fontsize=13, fontweight="bold")
ax2.set_xlabel("P&L (%)")
ax2.set_ylabel("Frequency")
ax2.legend()

fig.tight_layout()
plt.show()

### 5c. Monthly Performance

Bar chart of P&L per month with win rate overlay. Helps spot seasonality.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

colors = ["#4CAF50" if p >= 0 else "#F44336" for p in monthly["pnl"]]
x = range(len(monthly))
ax1.bar(x, monthly["pnl"], color=colors, alpha=0.7, label="P&L ($)")
ax1.set_ylabel("P&L ($)")
ax1.axhline(y=0, color="gray", linestyle="-", linewidth=0.5)

# Win rate on secondary axis
ax2 = ax1.twinx()
ax2.plot(list(x), monthly["win_rate"].values, "o-", color="#FF9800", linewidth=2, markersize=8, label="Win Rate %")
ax2.set_ylabel("Win Rate (%)", color="#FF9800")
ax2.set_ylim(0, 100)

ax1.set_xticks(list(x))
ax1.set_xticklabels(monthly["month"], rotation=45, ha="right")
ax1.set_title("Monthly Performance", fontsize=14, fontweight="bold")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

fig.tight_layout()
plt.show()

### 5d. Trade-by-Trade P&L (Waterfall)

Each bar is one trade — green for wins, red for losses. Helps you visually spot streaks.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = ["#4CAF50" if p > 0 else "#F44336" for p in df["pnl"]]
ax.bar(range(len(df)), df["pnl"], color=colors, alpha=0.8, width=0.8)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_title("Trade-by-Trade P&L", fontsize=14, fontweight="bold")
ax.set_xlabel("Trade #")
ax.set_ylabel("P&L ($)")

# Annotate the biggest win and biggest loss
best_idx = df["pnl"].idxmax()
worst_idx = df["pnl"].idxmin()
ax.annotate(f"Best: ${df.loc[best_idx, 'pnl']:,.0f}", xy=(best_idx, df.loc[best_idx, "pnl"]),
            fontsize=9, ha="center", va="bottom", color="green", fontweight="bold")
ax.annotate(f"Worst: ${df.loc[worst_idx, 'pnl']:,.0f}", xy=(worst_idx, df.loc[worst_idx, "pnl"]),
            fontsize=9, ha="center", va="top", color="red", fontweight="bold")

fig.tight_layout()
plt.show()

## Summary & Next Steps

Run this cell to get a quick verdict on your strategy.

In [ ]:
print("=" * 60)
print("  STRATEGY VERDICT")
print("=" * 60)

checks = {
    "EV per trade > 0": ev_per_trade > 0,
    "Profit factor > 1.0": profit_factor > 1.0,
    "Reward-to-Risk > 1.0": reward_to_risk > 1.0,
    "Win rate > break-even": win_rate > breakeven_wr,
    "Max drawdown < 20%": (max_dd * 100) < 20,
    "Sharpe > 0.5": sharpe > 0.5,
    "Max consec losses <= 5": max_consec_losses <= 5,
}

for check, passed in checks.items():
    icon = "PASS" if passed else "FAIL"
    print(f"  [{icon}] {check}")

passed_count = sum(checks.values())
total_checks = len(checks)
print(f"\n  Score: {passed_count}/{total_checks}")

if passed_count >= 5:
    print("  >> Strategy shows promise. Consider forward-testing.")
elif passed_count >= 3:
    print("  >> Mixed results. Needs optimization before live trading.")
else:
    print("  >> Strategy needs significant work. Focus on improving R:R or reducing losses.")